# Time Series Validation

Validates Week 4 outputs: daily sentiment aggregation, moving averages, change points, Prophet forecasts, and topic-specific sentiment trends.

In [ ]:
import sys
sys.path.insert(0, '..')

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

DB_PATH = '../historical_reddit_data.db'

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print('Connected to', DB_PATH)

## 1. Daily Sentiment per Subreddit

In [ ]:
daily = pd.read_sql_query(
    "SELECT * FROM sentiment_daily ORDER BY date", conn
)
daily['date'] = pd.to_datetime(daily['date'])
print(f"Rows: {len(daily):,}  |  Subreddits: {daily['subreddit'].nunique()}")
daily.head()

In [ ]:
ma = pd.read_sql_query(
    "SELECT * FROM sentiment_moving_avg ORDER BY date", conn
)
ma['date'] = pd.to_datetime(ma['date'])

subreddits = daily['subreddit'].unique()
fig, axes = plt.subplots(len(subreddits), 1, figsize=(14, 3 * len(subreddits)), sharex=True)
if len(subreddits) == 1:
    axes = [axes]

for ax, sub in zip(axes, subreddits):
    d = daily[daily['subreddit'] == sub]
    m = ma[ma['subreddit'] == sub]
    ax.plot(d['date'], d['mean_score'], alpha=0.3, color='steelblue', label='Daily')
    ax.plot(m['date'], m['rolling_7d'],  color='steelblue', label='7-day MA')
    ax.plot(m['date'], m['rolling_30d'], color='darkorange', linestyle='--', label='30-day MA')
    ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
    ax.set_title(f'r/{sub}')
    ax.set_ylabel('Sentiment')
    ax.legend(loc='upper left', fontsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
fig.suptitle('Daily Sentiment with Moving Averages', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Change Points

In [ ]:
cp = pd.read_sql_query("SELECT * FROM change_points ORDER BY date", conn)
cp['date'] = pd.to_datetime(cp['date'])
print(f"Change points detected: {len(cp)}")

if not cp.empty:
    fig, axes = plt.subplots(len(subreddits), 1, figsize=(14, 3 * len(subreddits)), sharex=True)
    if len(subreddits) == 1:
        axes = [axes]

    for ax, sub in zip(axes, subreddits):
        d = daily[daily['subreddit'] == sub]
        c = cp[cp['subreddit'] == sub]
        ax.plot(d['date'], d['mean_score'], color='steelblue', alpha=0.6)
        for _, row in c.iterrows():
            color = 'red' if row['magnitude'] < 0 else 'green'
            ax.axvline(row['date'], color=color, linewidth=1.5, alpha=0.8,
                       label=f"Δ{row['magnitude']:+.2f}")
        ax.set_title(f'r/{sub} — change points')
        ax.set_ylabel('Sentiment')
        ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    fig.suptitle('Sentiment with PELT Change Points', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No change points in the database yet.')

## 3. Prophet Forecast

In [ ]:
fc = pd.read_sql_query("SELECT * FROM sentiment_forecast ORDER BY date", conn)
fc['date'] = pd.to_datetime(fc['date'])
print(f"Forecast rows: {len(fc)}")

if not fc.empty:
    fig, axes = plt.subplots(len(subreddits), 1, figsize=(14, 3 * len(subreddits)), sharex=False)
    if len(subreddits) == 1:
        axes = [axes]

    for ax, sub in zip(axes, subreddits):
        d = daily[daily['subreddit'] == sub]
        f = fc[fc['subreddit'] == sub]
        ax.plot(d['date'], d['mean_score'], color='steelblue', label='Historical')
        if not f.empty:
            ax.plot(f['date'], f['yhat'], color='darkorange', label='Forecast')
            ax.fill_between(
                f['date'], f['yhat_lower'], f['yhat_upper'],
                alpha=0.2, color='darkorange', label='95% CI'
            )
        ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
        ax.set_title(f'r/{sub}')
        ax.set_ylabel('Sentiment')
        ax.legend(loc='upper left', fontsize=8)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

    fig.suptitle('Prophet Sentiment Forecast (14 days ahead)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No forecast data yet — run scripts/run_timeseries.py first.')

## 4. Topic Sentiment Trends Heatmap

In [ ]:
tst = pd.read_sql_query(
    "SELECT topic_id, date, rolling_7d FROM topic_sentiment_trends ORDER BY date", conn
)
tst['date'] = pd.to_datetime(tst['date'])
print(f"Topic trend rows: {len(tst)}  |  Topics: {tst['topic_id'].nunique()}")

if not tst.empty:
    pivot = tst.pivot_table(index='topic_id', columns='date', values='rolling_7d')
    # Limit to top 30 topics by row count for readability
    top_topics = tst['topic_id'].value_counts().head(30).index
    pivot = pivot.loc[pivot.index.isin(top_topics)]

    fig, ax = plt.subplots(figsize=(16, max(6, len(pivot) * 0.35)))
    sns.heatmap(
        pivot,
        cmap='RdYlGn',
        center=0,
        vmin=-1,
        vmax=1,
        ax=ax,
        cbar_kws={'label': 'Sentiment (7-day rolling avg)'},
        xticklabels=max(1, len(pivot.columns) // 20),
    )
    ax.set_title('Topic Sentiment Over Time (top 30 topics)', fontsize=13)
    ax.set_xlabel('Date')
    ax.set_ylabel('Topic ID')
    plt.tight_layout()
    plt.show()
else:
    print('No topic trend data yet — run scripts/run_timeseries.py first.')

In [ ]:
conn.close()
print('Done.')